In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import numpy as np
import os
import random
import cv2

DATA_DIR = "data"
VIDEOS_CSV        = os.path.join(DATA_DIR, "manifests", "videos.csv")
FRAMES_DIR        = os.path.join(DATA_DIR, "extracted_frames")
FINAL_DATASET_DIR = os.path.join(DATA_DIR, "final_dataset")
SPLITS_CSV        = os.path.join(FINAL_DATASET_DIR, "split_assignments.csv")
IMAGES_DIR        = os.path.join(FINAL_DATASET_DIR, "images")

videos_df = pd.read_csv(VIDEOS_CSV)
splits_df = pd.read_csv(SPLITS_CSV)

# Resolve full image paths
splits_df['full_path'] = splits_df['filepath'].apply(
    lambda p: os.path.join(FINAL_DATASET_DIR, p)
)

print("=" * 50)
print("  MANIFEST STATS (videos.csv)")
print("=" * 50)
ok      = (videos_df['status'] == 'ok').sum()
failed  = (videos_df['status'] == 'failed').sum()
skipped = (videos_df['status'] == 'skipped').sum()
total_hrs = videos_df.loc[videos_df['status'] == 'ok', 'duration_sec'].sum() / 3600
print(f"  Total in manifest      : {len(videos_df)}")
print(f"  Successfully downloaded: {ok}")
print(f"  Failed                 : {failed}")
print(f"  Skipped                : {skipped}")
print(f"  Total content duration : {total_hrs:.2f} hours")

print()
print("=" * 50)
print("  FINAL DATASET STATS (split_assignments.csv)")
print("=" * 50)
print(f"  Total labeled samples  : {len(splits_df):,}")
print(f"  Unique labels          : {splits_df['label'].nunique()}")
print(f"  Unique videos          : {splits_df['video_id'].nunique()}")
print(f"  Unique frames          : {splits_df['frame_id'].nunique():,}")
print(f"  Splits                 : {sorted(splits_df['split'].unique())}")

In [ ]:
# --- Class Distribution & Split Breakdown ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Overall label distribution
label_counts = splits_df['label'].value_counts()
sns.barplot(x=label_counts.values, y=label_counts.index, palette='viridis', ax=axes[0])
axes[0].set_title("Overall Label Distribution")
axes[0].set_xlabel("Count")
for i, v in enumerate(label_counts.values):
    axes[0].text(v + 5, i, str(v), va='center', fontsize=8)

# 2. Train / Val / Test split sizes
split_counts = splits_df['split'].value_counts().reindex(['train', 'val', 'test'])
bars = axes[1].bar(split_counts.index, split_counts.values, color=['steelblue', 'orange', 'green'])
axes[1].set_title("Samples per Split")
axes[1].set_ylabel("Count")
for bar, val in zip(bars, split_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f"{val:,}", ha='center', fontsize=10, fontweight='bold')

# 3. Label distribution per split (stacked bar)
split_label = splits_df.groupby(['split', 'label']).size().unstack(fill_value=0)
split_label = split_label.reindex(['train', 'val', 'test'])
split_label.plot(kind='bar', stacked=True, ax=axes[2], colormap='tab20', legend=False)
axes[2].set_title("Label Distribution per Split (Stacked)")
axes[2].set_xlabel("Split")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)

plt.tight_layout()
plt.savefig("class_and_split_distribution.png", bbox_inches='tight')
plt.show()

# Print split % breakdown
print("\nSplit proportions:")
print(splits_df['split'].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
# --- Per-Video Contribution & Frames per Video ---
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Samples per video (sorted)
video_counts = splits_df.groupby('video_id').size().sort_values(ascending=False)
video_counts.plot(kind='bar', color='coral', ax=axes[0])
axes[0].set_title("Labeled Samples per Video")
axes[0].set_xlabel("Video ID")
axes[0].set_ylabel("Sample Count")
axes[0].tick_params(axis='x', rotation=90, labelsize=6)

# Unique frames used per video
frame_counts = splits_df.groupby('video_id')['frame_id'].nunique().sort_values(ascending=False)
frame_counts.plot(kind='bar', color='teal', ax=axes[1])
axes[1].set_title("Unique Frames Used per Video")
axes[1].set_xlabel("Video ID")
axes[1].set_ylabel("Frame Count")
axes[1].tick_params(axis='x', rotation=90, labelsize=6)

plt.tight_layout()
plt.savefig("per_video_stats.png", bbox_inches='tight')
plt.show()

# Avg squares labeled per frame
avg_sq_per_frame = splits_df.groupby('frame_id').size().mean()
print(f"Avg labeled squares per frame: {avg_sq_per_frame:.1f}")
print(f"\nTop 5 videos by sample count:\n{video_counts.head()}")

In [ ]:
# --- Spatial Distribution: Piece Heatmaps on the Board ---
# square_index: 0=a8 (top-left) to 63=h1 (bottom-right), row-major
def sq_to_rowcol(sq_idx):
    return sq_idx // 8, sq_idx % 8  # row 0=rank8, col 0=fileA

labels_to_plot = [l for l in splits_df['label'].unique() if l != 'Empty']
ncols = 4
nrows = (len(labels_to_plot) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for i, label in enumerate(sorted(labels_to_plot)):
    board = np.zeros((8, 8))
    subset = splits_df[splits_df['label'] == label]
    for sq in subset['square_index']:
        r, c = sq_to_rowcol(sq)
        board[r, c] += 1
    sns.heatmap(board, ax=axes[i], cmap='YlOrRd', cbar=True,
                xticklabels=['a','b','c','d','e','f','g','h'],
                yticklabels=['8','7','6','5','4','3','2','1'])
    axes[i].set_title(label, fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Piece Spatial Distribution on the Board", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("piece_heatmaps.png", bbox_inches='tight')
plt.show()

In [ ]:
# --- Example Images per Class ---
all_labels = sorted(splits_df['label'].unique())
n_examples = 6
ncols = n_examples
nrows = len(all_labels)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.5, nrows * 1.6))

for r, label in enumerate(all_labels):
    label_df = splits_df[splits_df['label'] == label]
    samples = label_df.sample(n=min(n_examples, len(label_df)), random_state=42)
    for c, (_, row) in enumerate(samples.iterrows()):
        ax = axes[r, c]
        img = mpimg.imread(row['full_path'])
        ax.imshow(img)
        ax.axis('off')
        if c == 0:
            ax.set_ylabel(label, rotation=0, labelpad=60, fontsize=8, va='center')
    for c in range(len(samples), ncols):
        axes[r, c].axis('off')

plt.suptitle("Example Images per Class (6 random samples)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("example_images_per_class.png", bbox_inches='tight')
plt.show()

In [ ]:
# --- Image Quality: Brightness & Size Distribution ---
sample_n = min(2000, len(splits_df))
sample_df = splits_df.sample(n=sample_n, random_state=42)

brightnesses = []
widths, heights = [], []

for _, row in sample_df.iterrows():
    img = cv2.imread(row['full_path'])
    if img is not None:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightnesses.append(gray.mean())
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(brightnesses, bins=40, kde=True, color='purple', ax=axes[0])
axes[0].set_title(f"Mean Pixel Brightness (n={len(brightnesses)})")
axes[0].set_xlabel("Avg Brightness (0=Black, 255=White)")

axes[1].scatter(widths, heights, alpha=0.3, s=10, color='steelblue')
axes[1].set_title("Image Dimensions (W x H)")
axes[1].set_xlabel("Width (px)")
axes[1].set_ylabel("Height (px)")

# Brightness per label (box plot)
label_brightness = []
for label in all_labels:
    subset = splits_df[splits_df['label'] == label].sample(n=min(200, len(splits_df[splits_df['label']==label])), random_state=0)
    for _, row in subset.iterrows():
        img = cv2.imread(row['full_path'], cv2.IMREAD_GRAYSCALE)
        if img is not None:
            label_brightness.append({'label': label, 'brightness': img.mean()})

bright_df = pd.DataFrame(label_brightness)
bright_df.boxplot(column='brightness', by='label', ax=axes[2], rot=90, fontsize=7)
axes[2].set_title("Brightness Distribution per Label")
axes[2].set_xlabel("")
plt.suptitle("")

plt.tight_layout()
plt.savefig("image_quality.png", bbox_inches='tight')
plt.show()

print(f"Image size (most common): {pd.Series(list(zip(widths, heights))).value_counts().head(3).to_string()}")